# 03 load mysql

Carga dos dados no MySQL

In [16]:

%run 02_transform.ipynb

DIM_TEMPO: 1458 datas | 4 anos (2014-2017)
Produtos com categoria inconsistente: 0
DIM_PRODUTO: 1862 produtos | 3 categorias | 17 subcategorias
States únicos: 41
<StringArray>
[             'Alabama',              'Arizona',             'Arkansas',
           'California',             'Colorado',          'Connecticut',
             'Delaware', 'District of Columbia',              'Florida',
              'Georgia']
Length: 10, dtype: str
DIM_CLIENTE: 793 clientes |3 segmentos
 sk_envio modalidade_envio  prazo_medio_dias
        1     Second Class               3.2
        2   Standard Class               5.0
        3      First Class               2.2
        4         Same Day               0.0
FATO_VENDAS: 9,994 registros | Retornos: 800


In [ ]:
import sys
import os

sys.path.append(os.path.abspath('..'))

from config import engine
from sqlalchemy import text

print("Conexão pronta para uso.")

In [17]:
from sqlalchemy import create_engine, text
import sys, os

sys.path.insert(0, '..')
from config import DB_CONFIG

# Criar connection string
conn_str = (f"mysql+pymysql://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
            f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
            f"?charset=utf8mb4")

engine = create_engine(conn_str, echo=False)

# Testar conexão
with engine.connect() as conn:
    result = conn.execute(text('SELECT VERSION()')).fetchone()
    print(f'MySQL {result[0]} — conexão OK')

MySQL 8.0.46 — conexão OK


In [18]:
from sqlalchemy import text

# 1. Limpeza do banco para evitar duplicidade em novas tentativas
print('Limpando o banco de dados...')
with engine.begin() as conn:
    # Desativa checagem de FK para permitir o truncamento das tabelas
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 0;"))
    conn.execute(text("TRUNCATE TABLE FATO_VENDAS;"))
    conn.execute(text("TRUNCATE TABLE DIM_TEMPO;"))
    conn.execute(text("TRUNCATE TABLE DIM_PRODUTO;"))
    conn.execute(text("TRUNCATE TABLE DIM_CLIENTE;"))
    conn.execute(text("TRUNCATE TABLE DIM_ENVIO;"))
    conn.execute(text("SET FOREIGN_KEY_CHECKS = 1;"))

# 2. TRATAMENTO DE DADOS (Corte de strings para respeitar limites do MySQL)
# Garantimos que as colunas de texto não excedam o limite definido no banco
dim_produto['nome_produto'] = dim_produto['nome_produto'].astype(str).str.slice(0, 120)
dim_produto['categoria']    = dim_produto['categoria'].astype(str).str.slice(0, 30)
dim_produto['subcategoria'] = dim_produto['subcategoria'].astype(str).str.slice(0, 30)

dim_cliente['nome_cliente'] = dim_cliente['nome_cliente'].astype(str).str.slice(0, 80)
dim_cliente['cidade']       = dim_cliente['cidade'].astype(str).str.slice(0, 60)
dim_cliente['estado']       = dim_cliente['estado'].astype(str).str.slice(0, 30)
dim_cliente['regiao']       = dim_cliente['regiao'].astype(str).str.slice(0, 20)

dim_envio['modalidade_envio'] = dim_envio['modalidade_envio'].astype(str).str.slice(0, 20)

# 3. Função de Carga com tratamento de erros
def load_table(df, table_name, engine, if_exists='append', index=False):
    # O método 'multi' é mais rápido, mas requer cuidado com o tamanho das strings
    df.to_sql(table_name, engine, if_exists=if_exists, index=index, method='multi', chunksize=1000)
    with engine.connect() as conn:
        n = conn.execute(text(f'SELECT COUNT(*) FROM {table_name}')).scalar()
        print(f' {table_name:20}: {n:>7,} registros carregados')

# 4. Execução da Carga
print('\nCarregando dimensões...')
load_table(dim_tempo, 'DIM_TEMPO', engine)
load_table(dim_produto, 'DIM_PRODUTO', engine)
load_table(dim_cliente, 'DIM_CLIENTE', engine)
load_table(dim_envio, 'DIM_ENVIO', engine)

print('\nCarregando tabela fato...')
load_table(fato_final, 'FATO_VENDAS', engine)
print('\nCarga concluída!')

Limpando o banco de dados...

Carregando dimensões...


C:\Users\GDK\AppData\Local\Temp\ipykernel_10560\3601375223.py:31: UserWarning: The provided table name 'DIM_TEMPO' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(table_name, engine, if_exists=if_exists, index=index, method='multi', chunksize=1000)


 DIM_TEMPO           :   1,458 registros carregados


C:\Users\GDK\AppData\Local\Temp\ipykernel_10560\3601375223.py:31: UserWarning: The provided table name 'DIM_PRODUTO' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(table_name, engine, if_exists=if_exists, index=index, method='multi', chunksize=1000)
C:\Users\GDK\AppData\Local\Temp\ipykernel_10560\3601375223.py:31: UserWarning: The provided table name 'DIM_CLIENTE' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(table_name, engine, if_exists=if_exists, index=index, method='multi', chunksize=1000)
C:\Users\GDK\AppData\Local\Temp\ipykernel_10560\3601375223.py:31: UserWarning: The provided table name 'DIM_ENVIO' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.

 DIM_PRODUTO         :   1,862 registros carregados
 DIM_CLIENTE         :     793 registros carregados
 DIM_ENVIO           :       4 registros carregados

Carregando tabela fato...
 FATO_VENDAS         :   9,994 registros carregados

Carga concluída!


C:\Users\GDK\AppData\Local\Temp\ipykernel_10560\3601375223.py:31: UserWarning: The provided table name 'FATO_VENDAS' is not found exactly as such in the database after writing the table, possibly due to case sensitivity issues. Consider using lower case table names.
  df.to_sql(table_name, engine, if_exists=if_exists, index=index, method='multi', chunksize=1000)
